# SRAにおける「仮想ニューロン」創発検証実験

このノートブックでは、SRAモデルに対して**「8つの異なるドメイン × 5種類の類似タスク = 全40タスク」**を同時に学習させます。
学習後にルーターの重みを解析し、「複数のタスクで共有されるシナプス（共通コア）」と「特定のタスクでしか使われないシナプス（周辺・個別）」の**組み合わせのパス**が形成されるか、すなわち機能のまとまりである**「仮想ニューロン（Cell Assembly）」**が創発するかを検証します。

In [ ]:
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
import collections

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn numpy nbformat

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 40タスクの設計とデータジェネレータ

8つのドメイン（配列操作、算術演算、論理演算、パターン認識、置換暗号、統計・集計、フィルタリング、要素変換）それぞれに5つのタスクを定義します。
VOCABは `0`〜`9`の数字といくつかの中間記号、そして `<TASK_...>` の固有トークンを使用します。


In [ ]:
# トークンの定義
SYMBOLS = [str(i) for i in range(10)] + ["Y", "N", "(", ")", "-"]
PAD, BOS, EOS, SEP = 0, 1, 2, 3

# タスクのリスト
DOMAIN_TASKS = {
    "Sequence Ops": ["seq_copy", "seq_reverse", "seq_sort", "seq_shift_left", "seq_shift_right"],
    "Arithmetic": ["math_add", "math_sub", "math_mul", "math_max", "math_min"],
    "Boolean Logic": ["logic_and", "logic_or", "logic_xor", "logic_nand", "logic_nor"],
    "Pattern Recognition": ["pattern_parity", "pattern_palindrome", "pattern_paren", "pattern_allsame", "pattern_contains5"],
    "Substitution": ["cipher_a", "cipher_b", "cipher_c", "cipher_d", "cipher_e"],
    "Statistics": ["stat_count0", "stat_findmax", "stat_findmin", "stat_length", "stat_countunique"],
    "Filtering": ["filter_even", "filter_odd", "filter_gt5", "take_first3", "take_last3"],
    "Transformation": ["trans_add1", "trans_sub1", "trans_mul2", "trans_invert", "trans_zero"]
}

ALL_TASKS = [task for tasks in DOMAIN_TASKS.values() for task in tasks]

TOKENS = {
    "<PAD>": PAD, "<BOS>": BOS, "<EOS>": EOS, "<SEP>": SEP
}
for s in SYMBOLS:
    TOKENS[s] = len(TOKENS)
for t in ALL_TASKS:
    TOKENS[f"<TASK_{t.upper()}>"] = len(TOKENS)

ID2TOK = {v: k for k, v in TOKENS.items()}
VOCAB_SIZE = len(TOKENS)

def encode_symbols(symbols):
    return [TOKENS[str(s)] for s in symbols]

# 各タスクのデータ生成ロジック
CIPHER_MAPS = {
    "cipher_a": [3, 1, 4, 5, 9, 2, 6, 8, 7, 0],
    "cipher_b": [9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
    "cipher_c": [0, 2, 4, 6, 8, 1, 3, 5, 7, 9],
    "cipher_d": [1, 3, 5, 7, 9, 0, 2, 4, 6, 8],
    "cipher_e": [5, 6, 7, 8, 9, 0, 1, 2, 3, 4]
}

def make_sample(task, min_len=4, max_len=8):
    n = random.randint(min_len, max_len)
    task_token = [TOKENS[f"<TASK_{task.upper()}>"]]
    
    # 基本入力シーケンス (主に数字)
    seq = [random.randint(0, 9) for _ in range(n)]
    out = []
    
    # --- Sequence Ops ---
    if task == "seq_copy":
        out = seq[:]
    elif task == "seq_reverse":
        out = seq[::-1]
    elif task == "seq_sort":
        out = sorted(seq)
    elif task == "seq_shift_left":
        out = seq[1:] + [seq[0]] if n > 0 else []
    elif task == "seq_shift_right":
        out = [seq[-1]] + seq[:-1] if n > 0 else []
        
    # --- Arithmetic ---
    elif task.startswith("math_"):
        a, b = random.randint(0, 9), random.randint(0, 9)
        seq = [a, b]
        if task == "math_add": out = [(a + b) % 10]
        elif task == "math_sub": 
            val = a - b
            if val < 0: out = ["-", abs(val)]
            else: out = [val]
        elif task == "math_mul": out = [(a * b) % 10]
        elif task == "math_max": out = [max(a, b)]
        elif task == "math_min": out = [min(a, b)]
        
    # --- Boolean Logic ---
    elif task.startswith("logic_"):
        a, b = random.randint(0, 1), random.randint(0, 1)
        seq = [a, b]
        if task == "logic_and": out = [a & b]
        elif task == "logic_or": out = [a | b]
        elif task == "logic_xor": out = [a ^ b]
        elif task == "logic_nand": out = [1 - (a & b)]
        elif task == "logic_nor": out = [1 - (a | b)]
        
    # --- Pattern Recognition ---
    elif task == "pattern_parity":
        out = ["Y" if sum(seq) % 2 == 0 else "N"]
    elif task == "pattern_palindrome":
        out = ["Y" if seq == seq[::-1] else "N"]
    elif task == "pattern_paren":
        # 括弧のタスクだけ特別に入力を書き換える
        seq_str = [random.choice(["(", ")"]) for _ in range(n)]
        bal, ok = 0, True
        for s in seq_str:
            bal += 1 if s == "(" else -1
            if bal < 0: ok = False
        ok = ok and bal == 0
        return [BOS] + task_token + encode_symbols(seq_str) + [SEP], [TOKENS["Y"] if ok else TOKENS["N"], EOS]
    elif task == "pattern_allsame":
        out = ["Y" if len(set(seq)) <= 1 else "N"]
    elif task == "pattern_contains5":
        out = ["Y" if 5 in seq else "N"]
        
    # --- Substitution ---
    elif task.startswith("cipher_"):
        cmap = CIPHER_MAPS[task]
        out = [cmap[x] for x in seq]
        
    # --- Statistics ---
    elif task == "stat_count0":
        out = [seq.count(0)]
    elif task == "stat_findmax":
        out = [max(seq)]
    elif task == "stat_findmin":
        out = [min(seq)]
    elif task == "stat_length":
        out = [len(seq)]
    elif task == "stat_countunique":
        out = [len(set(seq))]
        
    # --- Filtering ---
    elif task == "filter_even":
        out = [x for x in seq if x % 2 == 0]
        if not out: out = ["N"]
    elif task == "filter_odd":
        out = [x for x in seq if x % 2 != 0]
        if not out: out = ["N"]
    elif task == "filter_gt5":
        out = [x for x in seq if x > 5]
        if not out: out = ["N"]
    elif task == "take_first3":
        out = seq[:3]
    elif task == "take_last3":
        out = seq[-3:]
        
    # --- Transformation ---
    elif task == "trans_add1":
        out = [(x + 1) % 10 for x in seq]
    elif task == "trans_sub1":
        out = [(x - 1) % 10 for x in seq]
    elif task == "trans_mul2":
        out = [(x * 2) % 10 for x in seq]
    elif task == "trans_invert":
        seq = [random.randint(0, 1) for _ in range(n)]
        out = [1 - x for x in seq]
    elif task == "trans_zero":
        out = [0 for _ in seq]
    else:
        raise ValueError(f"Unknown task: {task}")
        
    return [BOS] + task_token + encode_symbols(seq) + [SEP], encode_symbols(out) + [EOS]

def make_multitask_batch(tasks, batch_size, min_len, max_len):
    batch_tasks = [random.choice(tasks) for _ in range(batch_size)]
    pairs = [make_sample(t, min_len, max_len) for t in batch_tasks]
    max_x = max(len(x) for x, _ in pairs)
    max_y = max(len(y) for _, y in pairs)
    
    x = torch.full((batch_size, max_x), PAD, dtype=torch.long)
    y = torch.full((batch_size, max_y), PAD, dtype=torch.long)
    
    for i, (xi, yi) in enumerate(pairs):
        x[i, :len(xi)] = torch.tensor(xi)
        y[i, :len(yi)] = torch.tensor(yi)
        
    return x.to(device), y.to(device), batch_tasks

## 2. モデルの設定と学習

SRAモデルを定義し、マルチタスクで学習させます。タスク数が多い（40個）ため、シナプスの数を `32` 個と多めに設定します。

In [ ]:
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=VOCAB_SIZE,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=32,  # タスクが多いのでシナプスも増やす
    k=3,              # 各トークンが3つのシナプスにルーティングされる
    max_seq_len=32
)

model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128).to(device)
optimizer = make_optimizer(model, lr=0.005)

print("Model initialized. Vocabulary Size:", VOCAB_SIZE)

In [ ]:
# 学習ループ
print("Multitask Training on 40 Tasks started...")
model.train()

epochs = 1500  # タスクが多いので少し長めに学習
batch_size = 128

for epoch in range(epochs):
    x, y, batch_tasks = make_multitask_batch(ALL_TASKS, batch_size, min_len=4, max_len=8)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1), ignore_index=PAD)
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")

## 3. 仮想ニューロンの可視化と解析

学習後、各タスクごとにルーターがどのシナプスを選択したかを抽出します。これをドメインごとにまとめてヒートマップ化することで、シナプスの使われ方に共通性があるか（仮想ニューロンの創発）を確認します。

In [ ]:
def get_task_routing_vector(task, samples=10):
    model.eval()
    # 複数サンプルでルーティングの平均を計算
    x, y, _ = make_multitask_batch([task], batch_size=samples, min_len=6, max_len=6)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        _, routing_weights, _ = model(x, y_in)
    
    # Layer 0 の Routing Weights を使用
    # shape: [batch, seq_len, num_synapses]
    weights = routing_weights[0].mean(dim=0).mean(dim=0) # [num_synapses]
    return weights.cpu().numpy()

task_vectors = {}
for task in ALL_TASKS:
    task_vectors[task] = get_task_routing_vector(task, samples=50)

# 行列の構築
matrix = []
labels = []
domain_colors = []
palette = sns.color_palette("Set1", len(DOMAIN_TASKS))

for d_idx, (domain, tasks) in enumerate(DOMAIN_TASKS.items()):
    for task in tasks:
        matrix.append(task_vectors[task])
        labels.append(f"[{domain}] {task}")
        domain_colors.append(palette[d_idx])

matrix = np.array(matrix)

# ヒートマップ描画
plt.figure(figsize=(16, 12))
ax = sns.heatmap(matrix, cmap="YlGnBu", yticklabels=labels, xticklabels=range(config.num_synapses), cbar_kws={'label': 'Routing Probability'})
plt.title("Synapse Activation per Task (Emergence of Virtual Neurons)", fontsize=16)
plt.xlabel("Synapse ID", fontsize=14)
plt.ylabel("Task", fontsize=14)

# Y軸のラベルの色をドメインごとに変更
for i, tick_label in enumerate(ax.get_yticklabels()):
    tick_label.set_color(domain_colors[i])

plt.tight_layout()
plt.show()

In [ ]:
# 仮想ニューロン（同じシナプスの組み合わせを使うタスク群）の抽出と表示

virtual_neurons = collections.defaultdict(list)

for task, vector in task_vectors.items():
    # ルーティング確率が一定以上（ここでは0.1）のシナプスを「使用しているシナプス」とみなす
    active_synapses = tuple(sorted(np.where(vector >= 0.1)[0]))
    virtual_neurons[active_synapses].append(task)

# 含まれるタスクが多い順にソート
sorted_neurons = sorted(virtual_neurons.items(), key=lambda x: len(x[1]), reverse=True)

print("=== 抽出された仮想ニューロン（Cell Assemblies） ===")
print("※「同じシナプスの組み合わせ」を使用しているタスクを1つの仮想ニューロンとしてグループ化\n")

for i, (synapses, tasks) in enumerate(sorted_neurons, 1):
    print(f"[Virtual Neuron {i}]")
    print(f"  Core Synapses : {synapses}")
    print(f"  Belonging Tasks ({len(tasks)} tasks):")
    for t in tasks:
        print(f"    - {t}")
    print()


このように、**「完全に同じシナプスの組み合わせ」を使っているタスク群が、ネットワーク内で1つの独立した『仮想ニューロン』を形成している**ことが確認できます。類似タスクは同じ仮想ニューロンに属し、全く異なるタスクは別の仮想ニューロンを形成します。

### 考察ポイント
* **共通シナプスと個別シナプス**：多くのドメインで広く発火しているシナプス（共通コア）と、同じドメイン内のタスクで共通して強く発火しているシナプス（専門・個別）があるかを確認します。
* **仮想ニューロン（Cell Assembly）の形成**：それらの「共通シナプス」と「個別シナプス」の**発火の組み合わせパターン（まとまり）**が、そのドメイン・タスクを実行するための「仮想ニューロン」として機能していると言えます。
* **独立した発火**：置換暗号など全く異なるルールのタスクでは、他のタスクとは異なる独立したシナプス群の組み合わせが使われているかを観察してみてください。